<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch11_ex8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 11 - Exercise 8

a. Load CIFAR10. The dataset is composed of 60'000 32x32-pixel color images with 10 classes. Use `torchvision.datasets.CIFAR10`

In [2]:
# Use the code on p. 346

import torch
import torchvision
import torchvision.transforms.v2 as T
from google.colab import drive


drive.mount('/content/drive')
PERSISTENT_ROOT = "/content/drive/MyDrive/cifar10_data"

# preprocessing function to convert PIL images to tensors
# scale=True: pixels with values between 0 and 1
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.CIFAR10(
    root = PERSISTENT_ROOT, train = True, download = True, transform=toTensor)
test_data = torchvision.datasets.CIFAR10(
    root = PERSISTENT_ROOT, train = False, download = True, transform=toTensor)


Mounted at /content/drive


100%|██████████| 170M/170M [32:25<00:00, 87.6kB/s]


In [3]:
# we are going to use early stopping so let's create a validation set.
torch.manual_seed(42)
train_data, valid_data = torch.utils.data.random_split(
    train_and_valid_data, [45000, 5000])

In [4]:
# Let's have a look at the files
X_sample, y_sample = train_data[0]
print(f"Shape of training data: {X_sample.shape}")

# Extract the label from each of the first 40 samples
y_sample = [train_data[i][1] for i in range(40)]
print(f"The first 20 labels: {y_sample}")
    # we see that the labels are in {0, ... , 9}, so no need to shift them

Shape of training data: torch.Size([3, 32, 32])
The first 20 labels: [6, 2, 8, 6, 6, 9, 4, 5, 4, 4, 5, 5, 6, 6, 9, 7, 7, 8, 0, 8, 5, 2, 3, 3, 7, 6, 4, 4, 3, 2, 4, 5, 9, 9, 8, 1, 3, 6, 0, 9]


In [5]:
# Create the DataLoaders
from torch.utils.data import DataLoader

batch_size = 256

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size)

b. Build a DNN with 20 hidden layers of 100 neurons each (that's too many, but it's the point of this exercise). Use He initialization and the Swish activation function (using `nn.SiLU`). Since this is a classification task, you will need an output layer with one neuron per class.

In [14]:
import torch.nn as nn

# the following function is to be used as model.apply(use_he_init). p367
def use_he_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_uniform_(module.weight)
    nn.init.zeros_(module.bias)

# now let's create the model
# recall that nn. CrossEntropyLoss works directly with logits so this does not
# need an activation function in the output layer


class DeepModel(nn.Module):
  def __init__(self, n_inputs, n_hidden, n_neurons, n_classes):
    super().__init__()
    self.mlp=nn.Sequential(
        nn.Flatten(),
        nn.Linear(n_inputs, n_neurons), #e.g 54-100
        nn.SiLU(),
    )
    for _ in range(n_hidden-1):
      self.mlp.append(nn.Linear(n_neurons, n_neurons))
      self.mlp.append(nn.SiLU())
    self.mlp.append(nn.Linear(n_neurons, n_classes)) #e.g 100-10

  def forward(self, X):
    return self.mlp(X)

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [24]:
torch.manual_seed(42)
model = DeepModel(3*32*32,20,100,10)
model.apply(use_he_init)
model.to(device)

DeepModel(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3072, out_features=100, bias=True)
    (2): SiLU()
    (3): Linear(in_features=100, out_features=100, bias=True)
    (4): SiLU()
    (5): Linear(in_features=100, out_features=100, bias=True)
    (6): SiLU()
    (7): Linear(in_features=100, out_features=100, bias=True)
    (8): SiLU()
    (9): Linear(in_features=100, out_features=100, bias=True)
    (10): SiLU()
    (11): Linear(in_features=100, out_features=100, bias=True)
    (12): SiLU()
    (13): Linear(in_features=100, out_features=100, bias=True)
    (14): SiLU()
    (15): Linear(in_features=100, out_features=100, bias=True)
    (16): SiLU()
    (17): Linear(in_features=100, out_features=100, bias=True)
    (18): SiLU()
    (19): Linear(in_features=100, out_features=100, bias=True)
    (20): SiLU()
    (21): Linear(in_features=100, out_features=100, bias=True)
    (22): SiLU()
    (23): Linear(in_features=100, out_features=100, bi

c. Using NAdam optimization and early stopping, train the network on the CIFAR10 dataset. Remember to search for the right learning rate each time you change the model's architecture or hyperparameters.

In [9]:
pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 20.5 MB/s eta 0:00:00


In [26]:
# This evaluation function checks how our model performs on a given dataset
# From notes of Chapter 10
import torchmetrics

def evaluate_tm(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch) #update at each iteration
  return metric.compute()


In [27]:

import time
def train_with_early_stopping(model, optimizer, criterion, metric,
          train_loader, valid_loader, n_epochs,
            patience=10, checkpoint_path=None, scheduler = None):
  checkpoint_path = checkpoint_path or "my_checkpoint.pt"

  history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
  best_metric = 0.0 #we save the current best validation metric
  patience_counter = 0

  for epoch in range(n_epochs):

    # We MUST force the model back into training mode at the start of every epoch.
    model.train()

    # We wipe the metric memory clean before the new epoch starts.
    metric.reset()

    total_loss = 0
    t0=time.time()

    for X_batch, y_batch in train_loader:

      # Send data to GPU/CPU memory
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      # Forward pass: get model predictions
      y_pred = model(X_batch)

      # Calculate how wrong the model is (Loss value)
      loss = criterion(y_pred, y_batch)

      # We use '.item()' to extract the raw Python number from the loss tensor.
      total_loss += loss.item()

      # Backpropagation: Calculate gradients and update model parameters
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      # We feed the current batch's predictions and targets into the tracker.
      metric.update(y_pred, y_batch)

    # Calculate average loss for this training epoch
    mean_loss = total_loss / len(train_loader)
    history["train_losses"].append(mean_loss)
    train_metric = metric.compute().item()
    valid_metric = evaluate_tm(model, valid_loader, metric).item()
    if valid_metric > best_metric:
          torch.save(model.state_dict(), checkpoint_path)
          best_metric = valid_metric
          best = " (best)" #print (best) next to the validation metric if better
          patience_counter = 0
    else:
          patience_counter += 1
          best = ""

    t1 = time.time()
    history["train_losses"].append(total_loss / len(train_loader))
    history["train_metrics"].append(train_metric)
    history["valid_metrics"].append(valid_metric)
    print(f"Epoch {epoch + 1}/{n_epochs}, "
          f"train loss: {history['train_losses'][-1]:.4f}, "
          f"train metric: {history['train_metrics'][-1]:.4f}, "
          f"valid metric: {history['valid_metrics'][-1]:.4f}{best}"
          f" in {t1 - t0:.1f}s"
        )
    if scheduler is not None:
      # change the learning rate according to the scheduler's rule
          scheduler.step()
    if patience_counter >= patience:
            print("Early stopping!")
            break
  # Reload the highest-accuracy weights so the model doesn't
  # stick with the worse, overfitted weights from the final epoch.
  # Because we are 10 epochs past the best model

  model.load_state_dict(torch.load(checkpoint_path))
  return history

In [28]:
optimizer = torch.optim.NAdam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [29]:
n_epochs = 100
history = train_with_early_stopping(model, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.2349, train metric: 0.1421, valid metric: 0.1764 (best) in 12.6s
Epoch 2/100, train loss: 2.1167, train metric: 0.1885, valid metric: 0.2256 (best) in 12.7s
Epoch 3/100, train loss: 1.9928, train metric: 0.2404, valid metric: 0.2632 (best) in 12.6s
Epoch 4/100, train loss: 1.9709, train metric: 0.2528, valid metric: 0.2666 (best) in 12.1s
Epoch 5/100, train loss: 1.9489, train metric: 0.2613, valid metric: 0.2270 in 12.3s
Epoch 6/100, train loss: 1.9352, train metric: 0.2665, valid metric: 0.2396 in 12.6s
Epoch 7/100, train loss: 1.9218, train metric: 0.2763, valid metric: 0.2692 (best) in 12.7s
Epoch 8/100, train loss: 1.9203, train metric: 0.2741, valid metric: 0.2616 in 12.8s
Epoch 9/100, train loss: 1.9197, train metric: 0.2735, valid metric: 0.2290 in 12.4s
Epoch 10/100, train loss: 3.2620, train metric: 0.1390, valid metric: 0.1970 in 12.3s
Epoch 11/100, train loss: 2.0111, train metric: 0.2327, valid metric: 0.2374 in 12.5s
Epoch 12/100, train loss: 1.

d. Now try adding batch-norm and compare the learning curves: is it converging faster than before? Does it produce a better model? How does it affect training speed?

In [30]:
class DeepModel2(nn.Module):
  def __init__(self, n_inputs, n_hidden, n_neurons, n_classes):
    super().__init__()
    self.mlp=nn.Sequential(
        nn.Flatten(),
        nn.BatchNorm1d(n_inputs),
        nn.Linear(n_inputs, n_neurons), #e.g 54-100
        nn.SiLU(),
    )
    for _ in range(n_hidden-1):
      self.mlp.append(nn.BatchNorm1d(n_neurons))
      self.mlp.append(nn.Linear(n_neurons, n_neurons))
      self.mlp.append(nn.SiLU())

    self.mlp.append(nn.BatchNorm1d(n_neurons))
    self.mlp.append(nn.Linear(n_neurons, n_classes)) #e.g 100-10

  def forward(self, X):
    return self.mlp(X)

In [32]:
torch.manual_seed(42)
model2=DeepModel2(3*32*32,20,100, 10)
model2.apply(use_he_init)
model2.to(device)

DeepModel2(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): BatchNorm1d(3072, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Linear(in_features=3072, out_features=100, bias=True)
    (3): SiLU()
    (4): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): Linear(in_features=100, out_features=100, bias=True)
    (6): SiLU()
    (7): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): Linear(in_features=100, out_features=100, bias=True)
    (9): SiLU()
    (10): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (11): Linear(in_features=100, out_features=100, bias=True)
    (12): SiLU()
    (13): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): Linear(in_features=100, out_features=100, bias=True)
    (15): SiLU()
    (16): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_runnin

In [33]:
optimizer = torch.optim.NAdam(model2.parameters(), lr=0.002)
history2 = train_with_early_stopping(model2, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.1488, train metric: 0.2302, valid metric: 0.3202 (best) in 13.3s
Epoch 2/100, train loss: 1.7834, train metric: 0.3517, valid metric: 0.3872 (best) in 13.6s
Epoch 3/100, train loss: 1.6443, train metric: 0.4073, valid metric: 0.4198 (best) in 13.5s
Epoch 4/100, train loss: 1.5513, train metric: 0.4471, valid metric: 0.4464 (best) in 13.8s
Epoch 5/100, train loss: 1.4750, train metric: 0.4732, valid metric: 0.4570 (best) in 13.5s
Epoch 6/100, train loss: 1.4178, train metric: 0.4952, valid metric: 0.4682 (best) in 13.3s
Epoch 7/100, train loss: 1.3658, train metric: 0.5128, valid metric: 0.4818 (best) in 13.4s
Epoch 8/100, train loss: 1.3224, train metric: 0.5326, valid metric: 0.4822 (best) in 14.0s
Epoch 9/100, train loss: 1.2771, train metric: 0.5478, valid metric: 0.4826 (best) in 13.4s
Epoch 10/100, train loss: 1.2425, train metric: 0.5603, valid metric: 0.4848 (best) in 13.4s
Epoch 11/100, train loss: 1.2075, train metric: 0.5727, valid metric: 0.4804 in

e. Try replacing Swish with SELU, and make the necessary adjustments to ensure the network self-normalizes (i.e., standardize the input features, use LeCun normal initialization, make sure the DNN contains only a sequence of dense layers, without batch-norm, etc.).

In [34]:
class DeepModel_with_SELU(nn.Module):
  def __init__(self, n_inputs, n_hidden, n_neurons, n_classes):
    super().__init__()
    self.mlp=nn.Sequential(
        nn.Flatten(),
        nn.BatchNorm1d(n_inputs),
        nn.Linear(n_inputs, n_neurons), #e.g 54-100
        nn.SELU(),
    )
    for _ in range(n_hidden-1):
      self.mlp.append(nn.Linear(n_neurons, n_neurons))
      self.mlp.append(nn.SELU())

    self.mlp.append(nn.Linear(n_neurons, n_classes)) #e.g 100-10

  def forward(self, X):
    return self.mlp(X)

In [35]:
# the following function is to be used as model.apply(use_he_init). p367
def use_lecun_init(module):
  if isinstance(module, nn.Linear):
    nn.init.kaiming_normal_(module.weight, mode='fan_in', nonlinearity='linear')
    nn.init.zeros_(module.bias)

In [36]:
torch.manual_seed(42)
model_with_SELU=DeepModel_with_SELU(3*32*32,20,100, 10)
model_with_SELU.apply(use_lecun_init)
model_with_SELU.to(device)

DeepModel_with_SELU(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): BatchNorm1d(3072, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): Linear(in_features=3072, out_features=100, bias=True)
    (3): SELU()
    (4): Linear(in_features=100, out_features=100, bias=True)
    (5): SiLU()
    (6): Linear(in_features=100, out_features=100, bias=True)
    (7): SiLU()
    (8): Linear(in_features=100, out_features=100, bias=True)
    (9): SiLU()
    (10): Linear(in_features=100, out_features=100, bias=True)
    (11): SiLU()
    (12): Linear(in_features=100, out_features=100, bias=True)
    (13): SiLU()
    (14): Linear(in_features=100, out_features=100, bias=True)
    (15): SiLU()
    (16): Linear(in_features=100, out_features=100, bias=True)
    (17): SiLU()
    (18): Linear(in_features=100, out_features=100, bias=True)
    (19): SiLU()
    (20): Linear(in_features=100, out_features=100, bias=True)
    (21): SiLU()
    (22): Linear(in_features=100

In [37]:
optimizer = torch.optim.NAdam(model_with_SELU.parameters(), lr=0.002)
history_with_SELU = train_with_early_stopping(model_with_SELU, optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.0074, train metric: 0.2046, valid metric: 0.2692 (best) in 13.2s
Epoch 2/100, train loss: 1.8288, train metric: 0.2899, valid metric: 0.3272 (best) in 13.1s
Epoch 3/100, train loss: 1.7210, train metric: 0.3609, valid metric: 0.3914 (best) in 13.0s
Epoch 4/100, train loss: 1.6254, train metric: 0.4055, valid metric: 0.3902 in 13.2s
Epoch 5/100, train loss: 1.5403, train metric: 0.4375, valid metric: 0.4296 (best) in 12.5s
Epoch 6/100, train loss: 1.4637, train metric: 0.4703, valid metric: 0.4484 (best) in 12.4s
Epoch 7/100, train loss: 1.4120, train metric: 0.4921, valid metric: 0.4648 (best) in 12.0s
Epoch 8/100, train loss: 1.3616, train metric: 0.5156, valid metric: 0.4590 in 12.2s
Epoch 9/100, train loss: 1.3205, train metric: 0.5328, valid metric: 0.4826 (best) in 12.4s
Epoch 10/100, train loss: 1.2757, train metric: 0.5488, valid metric: 0.4836 (best) in 12.5s
Epoch 11/100, train loss: 1.2401, train metric: 0.5593, valid metric: 0.4808 in 12.4s
Epoch 1

f. Try regularizing the model with alpha dropout. Then, without retraining your model, see if you can achieve better accuracy using MC dropout.

In [46]:
class DeepModel_with_SELU_and_AlphaDropout(nn.Module):
  def __init__(self, n_inputs, n_hidden, n_neurons, n_classes, dropout_rate):
    super().__init__()
    self.mlp=nn.Sequential(
        nn.Flatten(),
        nn.BatchNorm1d(n_inputs),
        nn.AlphaDropout(dropout_rate),
        nn.Linear(n_inputs, n_neurons), #e.g 54-100
        nn.SELU(),
    )
    for _ in range(n_hidden-1):
      self.mlp.append(nn.AlphaDropout(dropout_rate))
      self.mlp.append(nn.Linear(n_neurons, n_neurons))
      self.mlp.append(nn.SELU())
    self.mlp.append(nn.AlphaDropout(dropout_rate))
    self.mlp.append(nn.Linear(n_neurons, n_classes)) #e.g 100-10

  def forward(self, X):
    return self.mlp(X)

In [48]:
torch.manual_seed(42)
model_with_SELU_and_AlphaDropout = DeepModel_with_SELU_and_AlphaDropout(3*32*32, 20, 100, 10, 0.1)
model_with_SELU_and_AlphaDropout.apply(use_lecun_init)
model_with_SELU_and_AlphaDropout.to(device)

DeepModel_with_SELU_and_AlphaDropout(
  (mlp): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): BatchNorm1d(3072, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): AlphaDropout(p=0.1, inplace=False)
    (3): Linear(in_features=3072, out_features=100, bias=True)
    (4): SELU()
    (5): AlphaDropout(p=0.1, inplace=False)
    (6): Linear(in_features=100, out_features=100, bias=True)
    (7): SELU()
    (8): AlphaDropout(p=0.1, inplace=False)
    (9): Linear(in_features=100, out_features=100, bias=True)
    (10): SELU()
    (11): AlphaDropout(p=0.1, inplace=False)
    (12): Linear(in_features=100, out_features=100, bias=True)
    (13): SELU()
    (14): AlphaDropout(p=0.1, inplace=False)
    (15): Linear(in_features=100, out_features=100, bias=True)
    (16): SELU()
    (17): AlphaDropout(p=0.1, inplace=False)
    (18): Linear(in_features=100, out_features=100, bias=True)
    (19): SELU()
    (20): AlphaDropout(p=0.1, inplace=False)
    (21): Linear(in_

In [51]:
optimizer = torch.optim.NAdam(model_with_SELU_and_AlphaDropout.parameters(), lr=0.001)
history_with_SELU_and_AlphaDropout = train_with_early_stopping(model_with_SELU_and_AlphaDropout,
                                                               optimizer, criterion, accuracy,
                                    train_loader, valid_loader, n_epochs)

Epoch 1/100, train loss: 2.2170, train metric: 0.1771, valid metric: 0.2440 (best) in 13.5s
Epoch 2/100, train loss: 1.9955, train metric: 0.2378, valid metric: 0.2622 (best) in 13.9s
Epoch 3/100, train loss: 1.9284, train metric: 0.2632, valid metric: 0.2864 (best) in 13.6s
Epoch 4/100, train loss: 1.8923, train metric: 0.2811, valid metric: 0.2932 (best) in 13.9s
Epoch 5/100, train loss: 1.8610, train metric: 0.3034, valid metric: 0.2936 (best) in 13.5s
Epoch 6/100, train loss: 1.8224, train metric: 0.3223, valid metric: 0.3436 (best) in 14.0s
Epoch 7/100, train loss: 1.7878, train metric: 0.3405, valid metric: 0.3528 (best) in 14.6s
Epoch 8/100, train loss: 1.7604, train metric: 0.3550, valid metric: 0.3678 (best) in 14.4s
Epoch 9/100, train loss: 1.7320, train metric: 0.3691, valid metric: 0.3872 (best) in 14.1s
Epoch 10/100, train loss: 1.7026, train metric: 0.3757, valid metric: 0.3962 (best) in 15.6s
Epoch 11/100, train loss: 1.6868, train metric: 0.3856, valid metric: 0.3954 in

In [56]:
model_with_SELU_and_AlphaDropout.eval()
for module in model_with_SELU_and_AlphaDropout.modules():
  if isinstance(module, nn.AlphaDropout):
    module.train()



def evaluate_mc(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch = X_batch.to(device)
      X_batch_repeated=X_batch.repeat_interleave(100, dim=0)
      y_batch = y_batch.to(device)
      y_logits = model(X_batch_repeated).reshape(len(X_batch), 100, 10)
      y_probas_all = torch.nn.functional.softmax(y_logits, dim = -1)
      y_probas = y_probas_all.mean(dim=1)
      metric.update(y_probas, y_batch) #update at each iteration
  return metric.compute()


torch.manual_seed(42)
final_validation_accuracy = evaluate_mc(model_with_SELU_and_AlphaDropout, valid_loader, accuracy)
print(f"The validation accuracy of the previous model using MC dropout: {final_validation_accuracy}")


The validation accuracy of the previous model using MC dropout: 0.4909999966621399
